In [1]:
from pathlib import Path

import duckdb

PROJECT_ROOT = Path.cwd().parent

assert (PROJECT_ROOT / "PROJECT_PLAN.md").exists(), (
    "Make sure this notebook is saved in the project's notebooks directory"
)

con = duckdb.connect()

con.read_parquet(
    str(PROJECT_ROOT / "data/processed/credit_card_training_all.parquet")
).create_view("training_cc_all", replace=True)

con.read_parquet(
    str(PROJECT_ROOT / "data/processed/credit_card_training_text.parquet")
).create_view("training_cc_text", replace=True)

print("Training Parquet views created.")

Training Parquet views created.


In [2]:
training_issue_sql = """
SELECT
    issue,
    COUNT(*) AS total_rows,
    COUNT(narrative) AS rows_with_narrative,
    ROUND(
        100.0 * COUNT(narrative) / COUNT(*),
        2
    ) AS narrative_coverage_pct,
    ROUND(
        100.0 * COUNT(narrative)
        / NULLIF(SUM(COUNT(narrative)) OVER (), 0),
        2
    ) AS share_of_available_text_pct
FROM training_cc_all
GROUP BY issue
ORDER BY rows_with_narrative DESC, issue
"""

training_issues = con.execute(training_issue_sql).df()

print(training_issues.to_string(index=False))
print("\nIssue categories:", len(training_issues))

assert int(training_issues["total_rows"].sum()) == 74957
assert int(training_issues["rows_with_narrative"].sum()) == 33117

                                                          issue  total_rows  rows_with_narrative  narrative_coverage_pct  share_of_available_text_pct
                Problem with a purchase shown on your statement       13684                 7791                   56.94                        23.53
                                          Getting a credit card        9241                 4042                   43.74                        12.21
                                               Fees or interest        6520                 3987                   61.15                        12.04
                             Other features, terms, or problems        6565                 3867                   58.90                        11.68
Problem with a company's investigation into an existing problem       10694                 2920                   27.31                         8.82
                                           Closing your account        4252                 2499    

In [3]:
training_duplicate_sql = r"""
WITH normalized AS (
    SELECT
        *,
        LOWER(
            REGEXP_REPLACE(narrative, '\s+', ' ', 'g')
        ) AS text_key
    FROM training_cc_text
),
text_groups AS (
    SELECT
        text_key,
        COUNT(*) AS group_size,
        COUNT(DISTINCT issue) AS distinct_issue_labels,
        COUNT(DISTINCT source_file) AS distinct_source_files,
        COUNT(
            DISTINCT DATE_TRUNC('month', received_date)
        ) AS distinct_months
    FROM normalized
    GROUP BY text_key
)
SELECT
    SUM(group_size) AS total_text_rows,
    COUNT(*) AS distinct_text_keys,
    COUNT(*) FILTER (
        WHERE group_size > 1
    ) AS repeated_text_groups,
    COALESCE(
        SUM(group_size) FILTER (WHERE group_size > 1),
        0
    ) AS rows_in_repeated_groups,
    SUM(group_size - 1) AS excess_repeated_rows,
    COUNT(*) FILTER (
        WHERE distinct_issue_labels > 1
    ) AS label_conflict_groups,
    COALESCE(
        SUM(group_size) FILTER (WHERE distinct_issue_labels > 1),
        0
    ) AS label_conflict_rows,
    COUNT(*) FILTER (
        WHERE distinct_source_files > 1
    ) AS repeated_groups_across_files,
    COUNT(*) FILTER (
        WHERE distinct_months > 1
    ) AS repeated_groups_across_months
FROM text_groups
"""

training_duplicates = con.execute(training_duplicate_sql).df()

display(training_duplicates.T.rename(columns={0: "value"}))

assert int(
    training_duplicates.iloc[0]["total_text_rows"]
) == 33117

,value
total_text_rows,33117.0
distinct_text_keys,30257.0
repeated_text_groups,468.0
rows_in_repeated_groups,3328.0
excess_repeated_rows,2860.0
label_conflict_groups,77.0
label_conflict_rows,992.0
repeated_groups_across_files,26.0
repeated_groups_across_months,108.0


In [4]:
audit_outputs = [
    ("training_issue_summary", training_issue_sql, training_issues),
    ("training_text_duplicates", training_duplicate_sql, training_duplicates),
]

for name, query, table in audit_outputs:
    (PROJECT_ROOT / "sql" / f"{name}.sql").write_text(
        query.strip() + "\n",
        encoding="utf-8",
    )

    table.to_csv(
        PROJECT_ROOT / "reports" / f"{name}.csv",
        index=False,
    )

print("Training text audit queries and reports saved.")

Training text audit queries and reports saved.


In [5]:
training_quality_sql = r"""
CREATE OR REPLACE VIEW training_text_quality AS
WITH normalized AS (
    SELECT
        *,
        LOWER(
            REGEXP_REPLACE(narrative, '\s+', ' ', 'g')
        ) AS text_key
    FROM training_cc_text
),
group_stats AS (
    SELECT
        text_key,
        COUNT(*) AS text_group_size,
        COUNT(DISTINCT issue) AS distinct_issue_labels,
        COUNT(DISTINCT source_file) AS distinct_source_files,
        COUNT(
            DISTINCT DATE_TRUNC('month', received_date)
        ) AS distinct_months
    FROM normalized
    GROUP BY text_key
)
SELECT
    t.complaint_id,
    t.issue,
    t.received_date,
    MD5(t.text_key) AS text_group_id,
    LENGTH(t.narrative) AS narrative_chars,
    g.text_group_size,
    g.distinct_issue_labels,
    g.distinct_source_files,
    g.distinct_months,
    g.text_group_size > 1 AS is_repeated_text,
    g.distinct_issue_labels > 1 AS has_label_conflict
FROM normalized AS t
JOIN group_stats AS g
    ON t.text_key = g.text_key
"""

con.execute(training_quality_sql)

training_quality_flags = con.execute("""
    SELECT *
    FROM training_text_quality
    ORDER BY received_date, complaint_id
""").df()

print("Quality flag rows:", len(training_quality_flags))

Quality flag rows: 33117


In [6]:
quality_check = con.execute("""
    SELECT
        COUNT(*) AS total_text_rows,
        COUNT(*) FILTER (
            WHERE is_repeated_text
        ) AS repeated_text_rows,
        COUNT(*) FILTER (
            WHERE has_label_conflict
        ) AS label_conflict_rows,
        COUNT(DISTINCT text_group_id) FILTER (
            WHERE has_label_conflict
        ) AS label_conflict_groups
    FROM training_text_quality
""").df()

display(quality_check.T.rename(columns={0: "value"}))

assert len(training_quality_flags) == 33117
assert training_quality_flags["complaint_id"].is_unique

training_quality_flags.to_parquet(
    PROJECT_ROOT / "data/processed/training_text_quality_flags.parquet",
    index=False,
)

quality_check.to_csv(
    PROJECT_ROOT / "reports/training_text_quality_summary.csv",
    index=False,
)

(PROJECT_ROOT / "sql/training_text_quality.sql").write_text(
    training_quality_sql.strip() + "\n",
    encoding="utf-8",
)

print("Training quality flags and summary saved.")

,value
total_text_rows,33117
repeated_text_rows,3328
label_conflict_rows,992
label_conflict_groups,77


Training quality flags and summary saved.


In [7]:
report = """# Training Text Audit

## Scope

The training period covers April 2024 through February 2025.

- All credit card complaints: 74,957
- Complaints with non-empty narratives: 33,117
- Missing issue labels: 0
- Issue categories: 15

## Class Distribution

The largest category contains 7,791 text records.
The smallest category contains 18.

Class imbalance remains substantial after expanding the date range.
Narrative availability also differs across categories.

## Exact Normalized Text Repetition

Matching lowercases text and collapses whitespace.

- Distinct text keys: 30,257
- Repeated text groups: 468
- Records in repeated groups: 3,328
- Excess repeated records: 2,860
- Conflicting-label groups: 77
- Records in conflicting-label groups: 992
- Repeated groups spanning source files: 26
- Repeated groups spanning months: 108

These checks do not identify all near-duplicate narratives.
Repeated text does not establish duplicate underlying complaint events.

## Current Status

Original labels and records are retained.
Row-level quality flags are stored locally.
No classifier has been trained.

The next step is near-duplicate investigation before constructing
the baseline training sample.
"""

(PROJECT_ROOT / "reports/training_text_audit.md").write_text(
    report,
    encoding="utf-8",
)

print("Training text audit report saved.")

Training text audit report saved.


In [8]:
sample_policy = """# Training Sample Policy

## Scope

These rules apply to text classification.
All complaint events remain available for aggregate business analysis.

## Labels

Retain the 15 observed issue categories initially.
Do not merge categories solely because they are rare.
Check category support again after sample filtering.
Do not revise the label scheme using final test outcomes.

## Planned Baseline Training Sample

Exclude training text groups with conflicting issue labels.
For consistent-label exact duplicate groups, retain the earliest record.
Break same-date ties deterministically using complaint ID.

This reduces repeated-text weighting without changing original records.
Construct a separate derived sample and record every exclusion.

Also plan a sensitivity comparison using all labeled training records,
so the effect of the filtering policy can be assessed.

## Near-Duplicate Text

Investigate near-duplicates using training data.
Document normalization, matching criteria, and reviewed examples.
Do not automatically delete records solely because texts are similar.
Finalize handling before fitting the baseline.

## Evaluation

Do not remove validation or test records based on their label conflicts.
Identify text overlap with training using documented text-based rules.
Report full-period performance and a separate unseen-text evaluation.
Report the coverage and class support of each evaluation population.

## Short Text

Retain substantive short narratives.
Do not apply an arbitrary minimum length filter.

## Status

This is the planned policy.
No baseline sample has been constructed yet.
"""

(PROJECT_ROOT / "docs/training_sample_policy.md").write_text(
    sample_policy,
    encoding="utf-8",
)

print("Training sample policy saved.")

Training sample policy saved.
